# EML Representation Demonstration

This notebook demonstrates the current EML transformation and validation workflow for the positive-domain exp/log fragment.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import sympy as sp
from eml import (
    evaluate_eml,
    total_node_count,
    tree_depth,
    unique_subexpression_count,
    validate_log_domain,
    validate_transformed_domain,
    transform,
    exp_log_rewrite,
)


## Transformation and structural comparison

Compare the original expression, its EML rewrite, and the exp/log baseline rewrite.

In [ ]:
x = sp.symbols('x')
expr = x * sp.exp(x) + sp.log(x + 2)
eml_expr = transform(expr)
baseline_expr = exp_log_rewrite(expr)

print('Original expression:')
print(expr)
print('
EML transformed:')
print(eml_expr)
print('
exp/log baseline:')
print(baseline_expr)

for label, node_expr in [('Original', expr), ('EML', eml_expr), ('exp/log', baseline_expr)]:
    print(f'\n{label} nodes: {total_node_count(node_expr)}; depth: {tree_depth(node_expr)}; unique subexpressions: {unique_subexpression_count(node_expr)}')


## Domain validation examples

Check symbolic domain safety for the original and transformed expressions on a sample interval.

In [ ]:
examples = [
    (sp.log(x), (0.1, 3.0)),
    (sp.log(x + 2), (0.0, 3.0)),
    (x * sp.exp(x), (-2.0, 2.0)),
]

for expr, (low, high) in examples:
    transformed = transform(expr)
    original_safe = validate_log_domain(expr, x, sample_points=[low, (low + high) / 2, high])
    transformed_safe = validate_transformed_domain(transformed, x, sample_points=[low, (low + high) / 2, high])
    print(f'Expression: {expr}')
    print(f'  Original log-domain safe: {original_safe}')
    print(f'  Transformed domain safe: {transformed_safe}')
    print()
